In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import os

# --- CONFIGURATION ---
# Pointing directly to your data location
DATA_PATH = r'D:\PJT2\CMAPSSData' 
PROCESSED_PATH = r'../../processed_data'

# The "Piecewise RUL" Cap (Models learn better when we cap max life)
RUL_CAP = 125 

# Window Size (The "Context" for the LSTM)
WINDOW_SIZE = 30 

# Columns in the NASA CMAPSS Dataset
cols = ['unit_number', 'time_in_cycles', 'setting_1', 'setting_2', 'setting_3', 's1', 's2', 's3',
        's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16',
        's17', 's18', 's19', 's20', 's21']

# --- STEP 1: LOAD DATA ---
print(f"Loading Data from {DATA_PATH}...")
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train_FD001.txt'), sep='\s+', header=None, names=cols)
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_FD001.txt'), sep='\s+', header=None, names=cols)
print(f"Train Shape: {train_df.shape}")
print(f"Test Shape: {test_df.shape}")

# --- STEP 2: CALCULATE RUL (Piecewise Linear) ---
def add_rul(df):
    # Get the max cycle for each unit
    max_cycle = df.groupby('unit_number')['time_in_cycles'].max().reset_index()
    max_cycle.columns = ['unit_number', 'max_cycle']
    
    # Merge back
    df = df.merge(max_cycle, on='unit_number', how='left')
    
    # Calculate RUL: Max - Current
    df['RUL'] = df['max_cycle'] - df['time_in_cycles']
    
    # Drop max_cycle column
    df = df.drop(columns=['max_cycle'])
    
    # CLIP RUL at 125 (The critical "Piecewise" step)
    df['RUL'] = df['RUL'].clip(upper=RUL_CAP)
    return df

train_df = add_rul(train_df)
print("RUL Calculated and Clipped at 125.")

# --- STEP 3: FEATURE SELECTION & SCALING ---
# Drop constant sensors (standard for FD001)
# Note: s1, s5, s10, s16, s18, s19 are constant in FD001
drop_labels = ['unit_number', 'time_in_cycles', 'setting_1', 'setting_2', 'setting_3', 
               's1', 's5', 's10', 's16', 's18', 's19', 'RUL']

# Features to Train on (Sensors 2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21)
features = train_df.columns.difference(drop_labels)

# Initialize Scaler
scaler = MinMaxScaler(feature_range=(0, 1))

# Fit on TRAIN, Transform both TRAIN and TEST
train_df[features] = scaler.fit_transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

print(f"Features Scaled: {list(features)}")

# --- STEP 4: SLIDING WINDOW GENERATOR ---
def gen_sequence(id_df, seq_length, seq_cols):
    """
    Reshapes: (Cycles, Features) -> (Samples, Window_Size, Features)
    """
    data_matrix = id_df[seq_cols].values
    num_elements = data_matrix.shape[0]
    
    # We simply iterate through the time series
    for start, stop in zip(range(0, num_elements-seq_length), range(seq_length, num_elements)):
        yield data_matrix[start:stop, :]

def gen_labels(id_df, seq_length, label):
    """
    Gets the RUL for the END of the window
    """
    data_matrix = id_df[label].values
    num_elements = data_matrix.shape[0]
    
    # We want the RUL at the last step of the window
    return data_matrix[seq_length:]

# Generate TRAIN sequences
X_train, y_train = [], []

print("Generating Sliding Windows...")
for unit_id in train_df['unit_number'].unique():
    # Process each engine separately
    unit_data = train_df[train_df['unit_number'] == unit_id]
    
    # Create windows
    for seq in gen_sequence(unit_data, WINDOW_SIZE, features):
        X_train.append(seq)
    
    # Create labels
    for label in gen_labels(unit_data, WINDOW_SIZE, 'RUL'):
        y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"Final X_train Shape: {X_train.shape}") # Expecting approx (20631, 30, 14)
print(f"Final y_train Shape: {y_train.shape}")

# --- STEP 5: SAVE PROCESSED DATA ---
if not os.path.exists(PROCESSED_PATH):
    os.makedirs(PROCESSED_PATH)

np.save(os.path.join(PROCESSED_PATH, 'X_train_FD001.npy'), X_train)
np.save(os.path.join(PROCESSED_PATH, 'y_train_FD001.npy'), y_train)

print(f"Files saved to {PROCESSED_PATH}")

Loading Data from D:\PJT2\CMAPSSData...
Train Shape: (20631, 26)
Test Shape: (13096, 26)
RUL Calculated and Clipped at 125.
Features Scaled: ['s11', 's12', 's13', 's14', 's15', 's17', 's2', 's20', 's21', 's3', 's4', 's6', 's7', 's8', 's9']
Generating Sliding Windows...
Final X_train Shape: (17631, 30, 15)
Final y_train Shape: (17631,)
Files saved to ../../processed_data


## Smoothing and Test Data prep

In [2]:
# --- STEP 6: SENSOR SMOOTHING (Exponential Moving Average) ---
# Why? Real sensors are noisy. Smoothing helps the LSTM see the "trend" not the "jitter".
def smooth_data(df, alpha=0.1):
    # Apply EMA only to the feature columns
    # alpha=0.1 means we trust the new value 10% and the old value 90% (Good for noise)
    smoothed = df.copy()
    smoothed[features] = smoothed[features].ewm(alpha=alpha).mean()
    return smoothed

print("Applying Exponential Smoothing...")
train_df = smooth_data(train_df)
test_df = smooth_data(test_df)
print("Sensors Smoothed.")

# --- STEP 7: PREPARE TEST DATA (Critical for Evaluation) ---
# For the TEST set, we only need to predict the RUL at the LAST time step.
# So we don't want sliding windows for the whole life. We just want the LAST 30 cycles.

def gen_test_sequence(id_df, seq_length, seq_cols):
    """
    Extracts ONLY the last sequence (Window Size 30) for each engine
    """
    data_matrix = id_df[seq_cols].values
    num_elements = data_matrix.shape[0]
    
    # If the engine has fewer than 30 cycles, we can't predict (Standard CMAPSS limitation)
    # But usually FD001 engines are long enough.
    if num_elements >= seq_length:
        return data_matrix[-seq_length:, :] # Take the last 30 rows
    else:
        return None

X_test = []
y_test = [] # The true RUL values from RUL_FD001.txt

# Load the True RULs for the test set
true_ruls = pd.read_csv(os.path.join(DATA_PATH, 'RUL_FD001.txt'), header=None, names=['RUL'])

print("Generating Test Sequences (Last 30 cycles per engine)...")

for unit_id in test_df['unit_number'].unique():
    unit_data = test_df[test_df['unit_number'] == unit_id]
    
    # Get the last 30 cycles
    seq = gen_test_sequence(unit_data, WINDOW_SIZE, features)
    
    if seq is not None:
        X_test.append(seq)
        # Get the true RUL for this engine from the file
        # Unit ID starts at 1, index starts at 0
        rul = true_ruls.iloc[unit_id - 1].values[0]
        y_test.append(rul)

X_test = np.array(X_test)
y_test = np.array(y_test)

print(f"Final X_test Shape: {X_test.shape}") # Should be (100, 30, 14) -> 1 window per engine
print(f"Final y_test Shape: {y_test.shape}") # Should be (100,)

# --- STEP 8: RE-SAVE WITH SMOOTHING & TEST DATA ---
np.save(os.path.join(PROCESSED_PATH, 'X_train_FD001.npy'), X_train) # Overwrite with smoothed version
np.save(os.path.join(PROCESSED_PATH, 'y_train_FD001.npy'), y_train)
np.save(os.path.join(PROCESSED_PATH, 'X_test_FD001.npy'), X_test)
np.save(os.path.join(PROCESSED_PATH, 'y_test_FD001.npy'), y_test)

print("Preprocessing Complete: Smoothed & Test Ready!")

Applying Exponential Smoothing...
Sensors Smoothed.
Generating Test Sequences (Last 30 cycles per engine)...
Final X_test Shape: (100, 30, 15)
Final y_test Shape: (100,)
Preprocessing Complete: Smoothed & Test Ready!
